In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import matthews_corrcoef, precision_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier
import pickle
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
import re
import itertools
import time
import warnings
warnings.filterwarnings("ignore", module="joblib")
import databento as db
import exchange_calendars as xcals

def close_times():

    # NYSE calendar
    cal = xcals.get_calendar("XNYS")

    # Build schedule for the date range you care about
    #start = "2018-01-01"
    #end = "2030-01-01"
    sched = cal.schedule.loc[:, ["open", "close"]].copy()

    # Convert to America/New_York
    sched["open_et"]  = sched["open"].dt.tz_convert("America/New_York")
    sched["close_et"] = sched["close"].dt.tz_convert("America/New_York")

    # Indicators
    #sched["is_trading_day"] = True
    sched["is_early_close"] = sched["close_et"].dt.time < pd.Timestamp("16:00", tz="America/New_York").time()

    # If you want a per-day close time (minutes since midnight ET)
    sched["session_duration"] = (sched["close_et"].dt.hour * 60 + sched["close_et"].dt.minute) - 9.5 * 60

    # Join to your intraday df by session date
    # assumes df has a Date column that is the NYSE session date (ET)
    sched_out = sched.reset_index().rename(columns={"index": "Date"})
    close_times_df = sched_out
    
    return close_times_df[['Date', 'close_et', 'is_early_close', 'session_duration']]

# Read the DBN file into a DBNStore object
dbn_store = db.DBNStore.from_file('qqq_1m.dbn')
# Convert the data to a pandas DataFrame for analysis
df = dbn_store.to_df()
df_main = df.reset_index()[['symbol', 'ts_event', 'close', 'open', 'high', 'low', 'volume']].copy()

# Add in session duration to account for early close on holidays
df_main['datetime_est'] = (df_main['ts_event'].dt.tz_convert('America/New_York'))
df_close_times = close_times()
# Merge close times with intraday data
df_main['Date'] = pd.to_datetime(df_main['datetime_est']).dt.strftime('%Y-%m-%d')
df_close_times["Date"] = pd.to_datetime(df_close_times["Date"]).dt.date
df_main["Date"] = pd.to_datetime(df_main["Date"]).dt.date
df_main = df_main.merge(df_close_times[['Date', 'session_duration']], on="Date", how="left")

# 1. Session Structure & Market Phases

In [21]:
def add_intraday_labels(df: pd.DataFrame, dt_col: str = "datetime_est") -> pd.DataFrame:
    
    out = df.copy()

    # Ensure datetime
    out[dt_col] = pd.to_datetime(out[dt_col], errors="coerce")
    if out[dt_col].isna().any():
        bad = out[dt_col].isna().sum()
        raise ValueError(f"{bad} rows in {dt_col} could not be parsed to datetime.")

    # Extract time-of-day in minutes since midnight (ET)
    tod_minutes = out[dt_col].dt.hour * 60 + out[dt_col].dt.minute
    out["_tod_minutes"] = tod_minutes

    # Open time (09:30 ET) in minutes
    premarket_min = 7 * 60  # 420
    open_min = 9 * 60 + 30  # 570
    #close_min = 16 * 60 - 1   # 960

    # Minutes since open (can be negative pre-market, positive post-open)
    out["rel_to_open"] = out["_tod_minutes"] - open_min
    out["rel_to_close"] = (open_min + out["session_duration"]) - out["_tod_minutes"]

    # Column 1: simple session label
    out["session_simple"] = np.select(
        [
            out["_tod_minutes"] < premarket_min,
            (out["_tod_minutes"] >= premarket_min) & (out["_tod_minutes"] < open_min),
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] <= (open_min + out["session_duration"])),
            out["_tod_minutes"] > (open_min + out["session_duration"]),
        ],
        ["overnight", "pre_market", "open_market", "post_market"],
        default=np.nan
    )

    # Column 2: detailed session label (your buckets)
    out["session_detail"] = np.select(
        [
            # Pre-market buckets
            (out["_tod_minutes"] < 7 *60),
            (out["_tod_minutes"] >= 7*60) & (out["_tod_minutes"] < 9*60),
            (out["_tod_minutes"] >= 9*60) & (out["_tod_minutes"] < open_min),

            # Open market buckets
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] < 9*60+45),
            (out["_tod_minutes"] >= 9*60+45) & (out["_tod_minutes"] < 10*60),
            (out["_tod_minutes"] >= 10*60) & (out["_tod_minutes"] < 12*60),
            (out["_tod_minutes"] >= 12*60) & (out["_tod_minutes"] < 14*60),
            (out["_tod_minutes"] >= 14*60) & (out["_tod_minutes"] < 15*60+30),
            (out["_tod_minutes"] >= 15*60+30) & (out["_tod_minutes"] < 15*60+45),
            (out["_tod_minutes"] >= 15*60+45) & (out["_tod_minutes"] <= (open_min + out["session_duration"])),

            # Post-market buckets
            (out["_tod_minutes"] > (open_min + out["session_duration"])) & (out["_tod_minutes"] < 16*60+15),
            (out["_tod_minutes"] >= 16*60+15) & (out["_tod_minutes"] < 17*60),
            (out["_tod_minutes"] >= 17*60) & (out["_tod_minutes"] <= 20*60),
        ],
        [
            "overnight",
            "early_pre_market",
            "late_pre_market",
            "early_open",
            "late_open",
            "morning",
            "midday",
            "late_day",
            "early_close",
            "late_close",
            "early_post_market",
            "late_post_market",
            "post_market",
        ],
        default="other"
    )

    # Cleanup
    out = out.drop(columns=["_tod_minutes", "_detail_simple_check"], errors="ignore")
    return out

#df_intraday_labels[df_intraday_labels['minutes_since_open'] == 390]
#Shortest minutes_since_open = -330 largest is 629. 0 = 930am, 389 = 4:00pm
df_intraday_labels = add_intraday_labels(df_main)
df_intraday_labels['Date'] = pd.to_datetime(df_intraday_labels['datetime_est']).dt.strftime('%Y-%m-%d')
df_labeled_final = df_intraday_labels[['symbol', 'datetime_est', 'rel_to_open', 'rel_to_close', 'session_simple', 
                    'session_detail', 'close', 'open', 'high', 'low', 'Date', 'session_duration']].copy()

# High Level Feature Engineering

In [33]:
df_features = df_labeled_final.dropna().copy()

# OC, HL, CH, CL ratios and magnitudes
o, h, l, c = (df_features[k].to_numpy() for k in ("open", "high", "low", "close"))

def dir_th(a, b, pct):
    return (a > (1 + pct) * b).astype(np.int8) - (a < (1 - pct) * b).astype(np.int8)

pairs = {
    "OC": (c, o),
    "HL": (h, l),
    "HC": (h, c),
    "LC": (c, l),
}

for k, (a, b) in pairs.items():
    df_features[f"{k}_Minute_Direction"] = np.sign(a - b).astype(np.int8)
    df_features[f"{k}_Minute_Magnitude"] = np.round(a / b - 1, 3)
    df_features[f"{k}_Minute_Direction_Low_TH"] = dir_th(a, b, 0.001)
    df_features[f"{k}_Minute_Direction_High_TH"] = dir_th(a, b, 0.01)

In [ ]:
# Percent of winning and losing minutes per session_simple and session_detail?
# 10 minute blocks? or start and end values for each session session_simple and session_detail?